MS MARCO v2: 1000 queries + BM25 candidates (not just qrels)

Train **qrels** are sparse on purpose (1–2 judged passages). The extra documents live in official **scoreddocs** (~100 BM25 hits per query).

This samples 1000 judged queries and keeps those BM25 hits. Unjudged hits get `relevance_label = 0`. Judged docs missing from the BM25 list are still added.

In [5]:
import random
from pathlib import Path

import ir_datasets
import pandas as pd
from tqdm import tqdm

RANDOM_SEED = 42
N_QUERIES = 1000
TOP_K = 100  # official v2 train scoreddocs are ~100 per query; lower this to 20 if you want Jusbrasil-sized slates
OUT_PATH = Path("data/msmarco_v2_1000q_bm25_candidates.csv")

In [6]:
dataset = ir_datasets.load("msmarco-passage-v2/train")

qrel_qids = []
seen = set()
for r in tqdm(dataset.qrels_iter(), desc="Collect query ids from qrels"):
    qid = str(r.query_id)
    if qid not in seen:
        seen.add(qid)
        qrel_qids.append(qid)

sample_ids = set(random.Random(RANDOM_SEED).sample(qrel_qids, N_QUERIES))
print("Eligible queries with qrels:", len(qrel_qids))
print("Sampled:", len(sample_ids))

Collect query ids from qrels: 284212it [00:00, 925180.89it/s]

Eligible queries with qrels: 277144
Sampled: 1000


In [7]:
qrel_rows = []
for r in tqdm(dataset.qrels_iter(), desc="Qrels for sampled queries"):
    qid = str(r.query_id)
    if qid in sample_ids:
        qrel_rows.append(
            {
                "query_id": qid,
                "doc_id": str(r.doc_id),
                "relevance_label": int(r.relevance),
            }
        )
qrels = pd.DataFrame(qrel_rows)
qrel_keys = set(zip(qrels["query_id"], qrels["doc_id"]))

cand_rows = []
per_query_count = {}
for s in tqdm(dataset.scoreddocs_iter(), desc="BM25 scoreddocs (full train file)"):
    qid = str(s.query_id)
    if qid not in sample_ids:
        continue
    n = per_query_count.get(qid, 0)
    if n >= TOP_K:
        continue
    per_query_count[qid] = n + 1
    cand_rows.append(
        {
            "query_id": qid,
            "doc_id": str(s.doc_id),
            "bm25_rank": n + 1,
            "bm25_score": float(s.score),
        }
    )

cands = pd.DataFrame(cand_rows)

# Keep judged docs even if they are missing from the BM25 list.
missing_pos = qrels.merge(cands[["query_id", "doc_id"]], on=["query_id", "doc_id"], how="left", indicator=True)
missing_pos = missing_pos.loc[missing_pos["_merge"] == "left_only", ["query_id", "doc_id"]]
missing_pos["bm25_rank"] = pd.NA
missing_pos["bm25_score"] = pd.NA
cands = pd.concat([cands, missing_pos], ignore_index=True)

df = cands.merge(qrels, on=["query_id", "doc_id"], how="left")
df["relevance_label"] = df["relevance_label"].fillna(0).astype(int)

query_text = {}
for q in tqdm(dataset.queries_iter(), desc="Query text"):
    qid = str(q.query_id)
    if qid in sample_ids:
        query_text[qid] = q.text
        if len(query_text) == N_QUERIES:
            break

df["query_text"] = df["query_id"].map(query_text)

print("Rows:", len(df))
print("Queries:", df["query_id"].nunique())
print("Docs per query:")
print(df.groupby("query_id").size().describe())
print("Positive labels:", int((df["relevance_label"] > 0).sum()))
df.head(3)

Qrels for sampled queries: 284212it [00:00, 995799.46it/s]
BM25 scoreddocs (full train file): 27713673it [00:31, 881388.66it/s]
/var/folders/9f/4brf4bwj2jg5n11qsp30715c0000gn/T/ipykernel_15171/92155223.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cands = pd.concat([cands, missing_pos], ignore_index=True)
Query text: 276377it [00:00, 1270185.24it/s]

Rows: 100681
Queries: 1000
Docs per query:
count    1000.00000
mean      100.68100
std         0.50941
min       100.00000
25%       100.00000
50%       101.00000
75%       101.00000
max       103.00000
dtype: float64
Positive labels: 1024


,query_id,doc_id,bm25_rank,bm25_score,relevance_label,query_text
0,696,msmarco_passage_34_402417032,1,12.4855,0,15 out of 16 as a percentage
1,696,msmarco_passage_34_402416716,2,11.9034,0,15 out of 16 as a percentage
2,696,msmarco_passage_04_155248602,3,11.5528,0,15 out of 16 as a percentage


In [8]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print("Wrote", OUT_PATH.resolve())

Wrote /Users/david/Documents/dev/llm-as-judge-for-ltr/notebooks/data/msmarco_v2_1000q_bm25_candidates.csv


In [ ]:
# Passage records have text (body) and a parent document id, not a title.
# Title comes from msmarco-document-v2 when that parent exists.
pass_store = ir_datasets.load("msmarco-passage-v2").docs_store()
doc_store = ir_datasets.load("msmarco-document-v2").docs_store()

unique_pass_ids = df["doc_id"].astype(str).unique().tolist()
passages = {}
for d in tqdm(
    pass_store.get_many_iter(unique_pass_ids),
    total=len(unique_pass_ids),
    desc="Fetch passage text",
):
    passages[str(d.doc_id)] = d

parent_ids = sorted(
    {
        str(d.msmarco_document_id)
        for d in passages.values()
        if getattr(d, "msmarco_document_id", None)
    }
)
parents = {}
for d in tqdm(
    doc_store.get_many_iter(parent_ids),
    total=len(parent_ids),
    desc="Fetch parent document titles",
):
    parents[str(d.doc_id)] = d


def passage_title(doc) -> str:
    parent = parents.get(str(getattr(doc, "msmarco_document_id", "") or ""))
    if parent is not None and getattr(parent, "title", None):
        return str(parent.title).strip()
    text = (doc.text or "").strip()
    first = text.split("\n", 1)[0].strip()
    return first if 8 <= len(first) <= 180 else ""


df["title"] = df["doc_id"].map(lambda i: passage_title(passages[i]) if i in passages else "")
df["body"] = df["doc_id"].map(lambda i: passages[i].text if i in passages else "")

print("Missing body:", int(df["body"].eq("").sum()))
print("Missing title:", int(df["title"].eq("").sum()))
df[["query_id", "doc_id", "title", "body"]].head(3)

                                                             
                                                                             [INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco_v2_passage.tar, you can symlink it here to avoid downloading it again: /Users/david/.ir_datasets/downloads/05946bac48a8ffee62e160213eab3fda
                                                             
                                                                             [INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco_v2_passage.tar
Fetch passage text:   0%|          | 0/99765 [00:00<?, ?it/s]






































































































































































































































































































































KeyboardInterrupt: 

In [ ]:
cols = [
    "query_id",
    "query_text",
    "doc_id",
    "title",
    "body",
    "bm25_rank",
    "bm25_score",
    "relevance_label",
]
df = df[cols]
df.to_csv(OUT_PATH, index=False)
print("Wrote", OUT_PATH.resolve(), "shape", df.shape)